# Python for AI — Class 6
### Functions: write it once, use it forever

You've spent five days learning Python's building blocks — variables, conditionals, loops, lists, tuples, dictionaries, sets. Today those blocks get packaged.

Today covers the foundations: defining functions, sending values back with `return`, every way to pass arguments, and scope — where your variables live. Tomorrow builds straight on top of it with lambda, recursion, decorators and generators.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

---
# 1. The problem functions solve

Three rectangles, three area calculations.

In [ ]:
print("Area of rectangle 1:", 4 * 5)
print("Area of rectangle 2:", 3 * 7)
print("Area of rectangle 3:", 10 * 2)

The calculation — `width * height` — is copy-pasted three times. Copy-pasted code is code you now have to fix in three places if the formula ever changes.

**A function lets you write the logic once, give it a name, and reuse it.**

---
# 2. Defining and calling a function

Notice the shape: a line ending in a **colon**, an **indented block** underneath. Same shape as `if`, `for`, and `while` — you already know how to read this.

In [ ]:
def area(width, height):
    return width * height

print(area(4, 5))
print(area(3, 7))
print(area(10, 2))

Break that first line down:

- `def` — "I am about to define a function"
- `area` — the name **you chose**, same rule as loop variable names on Day 3
- `(width, height)` — the **parameters**: named placeholders for values the function needs
- `:` then an indented block — the **body**, the code that runs every time you call it

`width` and `height` are **parameters** — the placeholders written in the `def` line. `4` and `5` in `area(4, 5)` are **arguments** — the actual values you hand over when you call it. People use the words loosely in conversation, but the distinction is worth having: a parameter is a name, an argument is a value.

Defining a function does not run it. Nothing gets printed until you **call** it — `area(4, 5)` — the same way `range(5)` on Day 3 did nothing on its own until a `for` loop consumed it.

---
# 3. `return` — sending a value back

`return` hands a value back to wherever the function was called, so you can store it, print it, or use it in another expression.

In [ ]:
def add(a, b):
    return a + b

result = add(3, 4)
print(result)
print(add(3, 4) * 10)

### The classic mix-up: `print` inside a function isn't the same as `return`

`print` just displays something on the screen — it doesn't hand anything back to the code that called the function.

In [ ]:
# WRONG ON PURPOSE - this looks like it works, but it hands back nothing
def add_and_print(a, b):
    print(a + b)      # displays the answer...

result = add_and_print(3, 4)
print("result is:", result)   # ...but result is None

`add_and_print` printed `7`, which looks like success — but `result` is `None`. A function with no `return` statement (or a `return` with nothing after it) **implicitly returns `None`**. If you need the value again later, you must `return` it, not just `print` it.

`return` also **exits the function immediately** — the moment Python hits a `return`, it leaves, skipping anything written after it. Same idea as `break` leaving a loop on Day 3.

In [ ]:
def check_sign(n):
    if n > 0:
        return "positive"
    if n < 0:
        return "negative"
    return "zero"

print(check_sign(5))
print(check_sign(-3))
print(check_sign(0))

---
# 4. Parameters are matched by position

When you call a function with plain values, they fill the parameters **left to right, in order**.

In [2]:
def greet(name, greeting):
    print(greeting, name)

greet("Ali", "Hello")

Hello Ali


In [3]:
# WRONG ON PURPOSE - the arguments are swapped, so the roles are swapped too
greet("Hello", "Ali")

Ali Hello


No error — `greeting` is now `"Ali"` and `name` is now `"Hello"`. Positional arguments only work if you get the order right; there's nothing checking that "Ali" was meant to be a name.

---
# 5. Default arguments

Give a parameter a value in the `def` line, and callers can skip it.

In [7]:
def greet(name, greeting="Hello"):
    print(greeting, name)

greet("Ali")               # uses the default
greet("Ali", "Salaam")     # overrides it

**A default parameter can't come before a required one** — Python needs to know, just by reading the `def` line, which arguments are optional.

In [ ]:
# BREAKS ON PURPOSE - a default parameter can't come before a required one
def greet(greeting="Hello", name):
    print(greeting, name)

### The mutable default trap

This is one of the most well-known gotchas in Python, and it connects straight back to Day 4's copy trap: a default value is created **once**, when the function is defined — not fresh on every call.

In [42]:
# WRONG ON PURPOSE - the same list is reused across every call
def add_item(item, cart=[]):
    cart.append(item)
    return cart

print(add_item("pen"))     # ['pen'] - looks right
print(add_item("book"))    # ['pen', 'book'] - wait, where did 'pen' come from?

['pen']
['pen', 'book']


`cart=[]` only runs **once**, at `def` time. Every call that doesn't supply its own `cart` shares that *same* list — exactly like `b = a` on Day 4 gave you a second sticker on the same list, not a fresh one.

The fix: default to `None`, and create the real empty list *inside* the function body, where it's rebuilt on every call.

In [5]:
def add_item(item, cart=None):
    if cart is None:
        cart = []
    cart.append(item)
    return cart

print(add_item("pen"))
print(add_item("book"))

['pen']
['book']


> **Rule of thumb:** never use a mutable value (`[]`, `{}`, `set()`) as a default argument. Use `None` and build the real thing inside the function.

---
# 6. Keyword arguments

Instead of relying on position, you can name each argument when you call the function. Order stops mattering, and the call documents itself.

In [ ]:
def describe_pet(name, animal_type, age):
    print(f"{name} is a {age}-year-old {animal_type}")

describe_pet("Rex", "dog", 3)                        # positional
describe_pet(name="Rex", animal_type="dog", age=3)   # keyword
describe_pet(age=3, name="Rex", animal_type="dog")   # order doesn't matter now

You can mix the two, but **every positional argument must come before every keyword argument** — once you start naming values, you can't go back to unnamed ones.

In [ ]:
describe_pet("Rex", age=3, animal_type="dog")     # fine - positional first, then keyword

In [ ]:
# BREAKS ON PURPOSE - a positional argument after a keyword argument
describe_pet(name="Rex", "dog", 3)

---
# 7. `*args` — any number of positional arguments

Sometimes you don't know in advance how many values a function will get. `*args` collects every extra positional argument into a **tuple**.

In [6]:
def total(*numbers):
    print(numbers, type(numbers))
    return sum(numbers)

print(total(1, 2, 3))
print(total(1, 2, 3, 4, 5))
print(total())

(1, 2, 3) <class 'tuple'>
6
(1, 2, 3, 4, 5) <class 'tuple'>
15
() <class 'tuple'>
0


`args` isn't a keyword — it's just a name, and the convention everyone uses. The `*` is what actually matters: it tells Python "gather up whatever positional arguments are left over." Same idea as Day 3's "the name is yours to pick, `for letter in name` just reads better than `for x in name`."

### Real use case: a POS till

A shop's point-of-sale (POS) till has no idea how many items the next customer will buy — one, or thirty. `*args` handles any number.

(You've actually been using `*args` since Day 1: `print()` and `max()` both accept any number of values.)

In [ ]:
def ring_up(*prices):
    subtotal = sum(prices)
    tax = subtotal * 0.18                 # 18% sales tax
    return round(subtotal + tax, 2)

print(ring_up(250))                       # a customer buying one item
print(ring_up(250, 1200, 80, 45))         # a customer with a full basket

---
# 8. `**kwargs` — any number of keyword arguments

`**kwargs` is the keyword-argument version: it collects every extra `name=value` pair into a **dictionary**.

In [7]:
def print_profile(**details):
    print(details, type(details))
    for key, value in details.items():
        print(key, "->", value)

print_profile(name="Ali", age=25, city="Karachi")

{'name': 'Ali', 'age': 25, 'city': 'Karachi'} <class 'dict'>
name -> Ali
age -> 25
city -> Karachi


That `.items()` loop is exactly Day 5's pattern — `**kwargs` is just a dictionary that arrived through a function call instead of a `{}` literal.

### Combining regular parameters, `*args`, and `**kwargs`

When you use all three, they must appear in this order in the `def` line: **regular parameters, then `*args`, then `**kwargs`.**

In [112]:
def order_summary(customer, *items, **extras):
    print("Customer:", customer)
    print("Items:", items)
    print("Extras:", extras.items())

order_summary("Ali", "pizza", "coke", discount=10, gift_wrap=True)

Customer: Ali
Items: ('pizza', 'coke')
Extras: dict_items([('discount', 10), ('gift_wrap', True)])


In [118]:
# BREAKS ON PURPOSE - **kwargs must be the last parameter
def broken_order(customer, *items, **extras):
    print(f"{customer}, {name}, {items}, {extras}")

broken_order("Walk in", 250, 300, "Sam",  discount=10)

Walk in, Ali, (250, 300, 'Sam'), {'discount': 10}


### Real use case: placing an online order

An e-commerce checkout has sensible defaults — standard shipping, no gift wrap — and each customer changes only the few options they care about. `**kwargs` lets one function accept any of those options without listing every possible one in the `def` line.

Many libraries you'll use later, including the AI libraries in Month 2, take optional keyword settings exactly like this.

In [125]:
def create_order(customer, *items, **options):
    
    order = {
        "customer": customer,
        "items": list(items),
        "shipping": "standard",               # defaults
        "gift_wrap": False,
    }
    for key, value in order.items():        # override or add whatever the customer chose
        order[key] = value
    return order

print(create_order("Ali", "Headphones"))
print(create_order("Fatima", "Laptop", "Mouse", shippings="express", gift_wrap=True, coupon="EID20"))

{'customer': 'Ali', 'items': ('Headphones',), 'shipping': 'standard', 'gift_wrap': False}
{'customer': 'Fatima', 'items': ('Laptop', 'Mouse'), 'shipping': 'standard', 'gift_wrap': False}


---
# 9. Scope: local vs. global

A variable created **inside** a function is **local** — it exists only while that function is running, and disappears the moment it returns.

In [127]:

def set_score():
    score = 100
    print("Inside the function:", score)
set_score()

NameError: name 'score' is not defined

In [128]:
# BREAKS ON PURPOSE - score only ever existed inside set_score
print("Outside the function:", score)

NameError: name 'score' is not defined

**NameError** — `score` was never created out here. This is a feature, not a bug: it means two different functions can both use a variable called `score` and never interfere with each other.

A variable created **outside** every function is **global**, and functions can freely *read* one.

In [130]:
# total_sales = 0

def show_sales():
    print("Sales so far:", total_sales)     # reading a global works fine

show_sales()

Sales so far: 0


**But assigning to a global-named variable inside a function creates a new local one instead** — Python decides a name is local for the *entire* function the moment it sees an assignment to it anywhere inside that function, even below where you tried to read it.

In [132]:
# BREAKS ON PURPOSE - Python treats total_sales as local because of the assignment below,
# so reading it on the right-hand side fails before that assignment ever runs
total_sales = 0

def add_sale(amount):
    total_sales = total_sales + amount
    print(total_sales)

add_sale(50)

UnboundLocalError: cannot access local variable 'total_sales' where it is not associated with a value

**UnboundLocalError.** To actually modify a global variable from inside a function, say so explicitly with `global`:

In [140]:
# global total_sales 
total_sales = 0

def add_sale(amount):
    global total_sales
    sum_of = total_sales + amount
    print(sum_of)

add_sale(50)
add_sale(30)
print("Final total:", total_sales)

50
30
Final total: 0


### Why you should reach for `global` rarely

`global` works, but treat it as a last resort. The idea: **don't let a function secretly reach out and change things outside itself. Hand it what it needs, and let it hand back the result.**

Think of a function as a chef in a kitchen:

- **The clean way (parameters + `return`):** you hand the chef the ingredients, and the chef hands you back a finished dish. You can see exactly what went in and what came out.
- **The `global` way:** the chef walks out of the kitchen, goes into your fridge, and changes what's in it without telling you. Later you open the fridge, something is missing, and you have no idea which chef did it or when.

Here's the same job done both ways.

In [141]:
# The global way - the function secretly changes something outside itself
total = 0

def add_sale(amount):
    global total
    total = total + amount

add_sale(50)          # nothing on this line tells you `total` just changed
print(total)

50


In [ ]:
# The clean way - what goes in and what comes out are both visible
def add_sale(total, amount):
    return total + amount

total = 0
total = add_sale(total, 50)    # takes total and 50, the result goes back into total
print(total)

Both print `50`. The difference is what you can see.

In the first version, reading `add_sale(50)` gives no hint that `total` changed — you'd have to open the function and read its insides to find out. In the second, the call itself shows everything: it takes `total` and `50`, and the result goes back into `total`.

**Why globals cause more bugs than they fix:** once several functions can all change the same global variable, a wrong value could have come from any of them, and you end up hunting through every function to find out who changed it. With parameters and `return`, each function only touches what you hand it — so when something goes wrong, there are far fewer places to look.

---
# 10. Your turn

**1.** Write a function `square(n)` that returns `n` squared. Print `square(5)` and `square(12)`.

**2.** Write a function `full_name(first, last)` that returns `"first last"`. Call it once with keyword arguments, in reversed order (`last=` before `first=`).

**3.** Write a function `power(base, exponent=2)` that returns `base` raised to `exponent`, defaulting to squaring. Call it once with just a base, and once with both arguments.

**4.** Write a function `total(*numbers)` that returns the sum of however many numbers you pass it. Test it with zero, one, and five numbers.

**5.** Predict, then run:
```python
def add_item(item, cart=[]):
    cart.append(item)
    return cart

print(add_item("pen"))
print(add_item("book"))
```

**6.** Find the bug:
```python
count = 0

def increment():
    count = count + 1
    return count

print(increment())
```

In [ ]:
# Your practice space

### Solutions

Try the exercises yourself first — these are one way to solve them.

In [ ]:
# 1. Square
def square(n):
    return n ** 2

print(square(5))
print(square(12))

In [ ]:
# 2. Full name, keyword arguments in reversed order
def full_name(first, last):
    return f"{first} {last}"

print(full_name(last="Khan", first="Ali"))

In [ ]:
# 3. power() with a default exponent
def power(base, exponent=2):
    return base ** exponent

print(power(5))
print(power(2, 10))

In [ ]:
# 4. total() with any number of arguments
def total(*numbers):
    return sum(numbers)

print(total())
print(total(7))
print(total(1, 2, 3, 4, 5))

In [ ]:
# 5. The mutable default trap - the SAME list is reused every call
def add_item(item, cart=[]):
    cart.append(item)
    return cart

print(add_item("pen"))     # ['pen']
print(add_item("book"))    # ['pen', 'book'] - not just ['book']

In [ ]:
# 6. Fix - reading a global that's also assigned to needs the `global` keyword
count = 0

def increment():
    global count
    count = count + 1
    return count

print(increment())
print(increment())

---
### Today you learned

- `def name(parameters):` defines a function; nothing runs until you **call** it
- `return` sends a value back and exits immediately; no `return` means the function silently gives back `None`
- Arguments are matched **by position**, unless you name them as **keyword arguments** — then order stops mattering
- Default arguments (`name="Hello"`) let callers skip a value — but **never default to a mutable value** like `[]`
- `*args` collects extra positional arguments into a tuple; `**kwargs` collects extra keyword arguments into a dictionary
- A variable created inside a function is **local** and disappears when the function returns; reading a global works, but *assigning* to one needs the `global` keyword — and it's usually better to pass values in and `return` them out

**Tomorrow:** lambda, recursion, decorators and generators — with real business examples.